# 01b. Tiền xử lý ảnh: OCR và mô tả ảnh bằng VLM

Chuyển thông tin trong ảnh thành text để PhoBERT dùng được:
- **OCR (PaddleOCR phát hiện vùng chữ + VietOCR nhận dạng)**: chữ xuất hiện trong ảnh (meme, ảnh chụp bình luận, biển báo...).
- **VLM (Vintern-1B)**: mô tả ngắn nội dung ảnh bằng tiếng Việt (ai/cái gì, hành động, bối cảnh, biểu cảm).

Chạy **một lần** cho toàn bộ train/public test/private test. Kết quả lưu JSON trong `paths.cache_dir`
(`ocr.json`, `vlm_description.json`). Bị ngắt thì chạy lại, ảnh đã xử lý sẽ được bỏ qua. Ảnh lỗi không được ghi
vào cache: script báo số ảnh còn thiếu và thoát với mã lỗi, chạy lại cell đó để thử lại.

Trên Kaggle: bật *Internet*, *GPU T4*, attach `hhhoang/vimmsd-dataset`. Chạy xong bấm *Save Version* để giữ
thư mục `/kaggle/working/cache`, rồi attach output này vào các notebook train và đặt `data.image_text.dir`
trỏ tới đó (hoặc `IMAGE_TEXT_DIR` trong notebook 07).

Prompt mô tả **không nhắc tới mỉa mai**, để VLM không đưa phán đoán nhãn vào input của model.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
!pip install -q paddlepaddle-gpu "paddleocr>=3.0" timm einops gdown
# vietocr ghim pillow/einops/gdown bản cũ: bỏ qua dependency để không hạ phiên bản của môi trường
!pip install -q --no-deps vietocr

In [ ]:
import gc
import pandas as pd
import torch
from PIL import Image
from vimmsd.data.dataset import load_all_records
from vimmsd.data.image_text import DESCRIBE_PROMPT, OCRExtractor, VLMDescriber, image_key

cfg = load_config("configs/base.yaml")
labels = list(cfg.data.labels)
records = load_all_records(cfg)
# 2 mẫu mỗi nhãn để xem thử chất lượng trước khi chạy toàn bộ
preview = [r for lab in range(len(labels)) for r in [x for x in records["train"] if x["label"] == lab][:2]]
print("prompt mô tả:", DESCRIBE_PROMPT)

## 1. OCR

Xem thử trên vài ảnh, sau đó chạy toàn bộ.

In [ ]:
ocr = OCRExtractor()
for r in preview:
    r["ocr"] = ocr(r["image_path"])
    print(f"[{labels[r['label']]}] {r['ocr'][:150]!r}")

In [ ]:
del ocr; gc.collect(); torch.cuda.empty_cache()
!python scripts/extract_image_text.py --config configs/base.yaml --task ocr

## 2. Mô tả ảnh bằng VLM

Xem tốc độ ở dòng tqdm để ước lượng thời gian cho ~13.700 ảnh. Nếu vượt quá 1 session Kaggle, chạy `--splits train` và `--splits public_test private_test` ở 2 session: cache được ghi tiếp, không mất phần đã chạy.

In [ ]:
vlm = VLMDescriber(backend="vintern")
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, r in zip(axes.flat, preview):
    r["description"] = vlm(r["image_path"])
    ax.imshow(Image.open(r["image_path"]).convert("RGB")); ax.axis("off")
    ax.set_title(f"[{labels[r['label']]}]\n{r['description'][:110]}", fontsize=8, wrap=True)
plt.tight_layout(); plt.show()

In [ ]:
del vlm; gc.collect(); torch.cuda.empty_cache()
!python scripts/extract_image_text.py --config configs/base.yaml --task description --vlm vintern

## 3. Kiểm tra kết quả

In [ ]:
cache_dir = Path(cfg.data.image_text.get("dir") or cfg.paths.cache_dir)
ocr_cache = json.loads((cache_dir / cfg.data.image_text.ocr_cache).read_text(encoding="utf-8"))
desc_cache = json.loads((cache_dir / cfg.data.image_text.description_cache).read_text(encoding="utf-8"))

rows = []
for split, recs in records.items():
    for r in recs:
        p = Path(r["image_path"])
        rows.append({"split": split, "label": labels[r["label"]] if r["label"] >= 0 else "(không nhãn)",
                     "caption": r["caption"], "ocr": ocr_cache.get(image_key(p.parent, p.name)),
                     "description": desc_cache.get(image_key(p.parent, p.name))})
df = pd.DataFrame(rows)
print("thiếu trong cache OCR:", df.ocr.isna().sum(), "| thiếu trong cache mô tả:", df.description.isna().sum())
df = df.fillna("")
df["có chữ (OCR)"] = df.ocr.str.len() > 0
df["số từ OCR"] = df.ocr.str.split().str.len()
df["số từ mô tả"] = df.description.str.split().str.len()
df["mô tả rỗng"] = df.description.str.len() == 0
df.groupby("label")[["có chữ (OCR)", "số từ OCR", "số từ mô tả", "mô tả rỗng"]].mean().round(2)

In [ ]:
# mô tả giống hệt nhau lặp lại nhiều lần thường là dấu hiệu VLM trả lời chung chung
display(df.description.value_counts().head(10))
df[df.split == "train"].sample(10, random_state=0)[["label", "caption", "ocr", "description"]]